# NestWatch nests in NAIP imagery: can we see the nest?

1. Pick one or more **species** (common name or 4-6 letter code) in the configuration cell.
2. Nests are drawn at random, and each is kept only if a **NAIP survey from the same year as the observation** covers it (looked up on Planetary Computer STAC, as in `search_naip` in [data.py](../../src/eagle_als/coreg/data.py)).
3. For each kept nest the notebook shows a true-color (optionally color-infrared) NAIP chip centered on the reported location, with a circle matching the 30 m lidar cookie radius used in `nestwatch_lidar_cookies.ipynb`.

NAIP is read as cloud-optimized GeoTIFF windows, so only the chip is downloaded. Chips are cached on disk, so re-running is fast. Run this notebook from `data_cleaning/data_exploration/` (the relative paths below assume that).

Verdict: nests not visible in NAIP, just as they are not visible in lidar (if these coords/times are correct)

## Configuration

In [ ]:
SPECIES = ['Osprey']#,'Great Horned Owl',"Bald Eagle"]   # list of common names ('Tree Swallow') and/or codes ('easblu'); see the species table below
N_SHOW = 8               # number of nests to show
SIZE = 100               # chip width in meters
RES = 0.6                # chip pixel size in meters (NAIP is 0.3-1 m depending on state/year)
CIRCLE_RADIUS = 30       # dashed circle radius in meters (30 = lidar cookie radius); None to hide
BANDS = 'rgb'            # 'rgb' (true color) or 'cir' (color infrared: NIR, red, green; vegetation shows red)
MIN_YEAR = 2010          # NAIP on Planetary Computer is only reliably available from about 2010
ONE_PER_LOCATION = True  # at most one attempt per NestWatch Location ID (nest boxes are reused every year)
PER_SPECIES = False      # True: N_SHOW nests for each selected species; False: N_SHOW nests in total
SEED = None              # int for reproducible draws, None for different ones every run
MAX_TRIES = 500          # give up after checking this many candidate nests for NAIP coverage
COLS = 4                 # chips per row
CSV = '../../datasets/raw/NestWatch/NestWatch Open Dataset/attempts_locs_20260120.csv'
CACHE_DIR = '~/.cache/eagle_naip_nestwatch'

In [ ]:
import hashlib
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

sys.path.append('../../src')
from eagle_als.coreg import data as D

rng = np.random.default_rng(SEED)
cache_dir = Path(CACHE_DIR).expanduser()
cache_dir.mkdir(parents=True, exist_ok=True)

## Load NestWatch attempts and choose species

The table below lists the species selected; any species in the data can be used.

In [ ]:
cols = ['Attempt ID', 'Location ID', 'Latitude', 'Longitude', 'Species Code', 'Species Name', 'Year', 'Subnational Code']
nw = pd.read_csv(CSV, usecols=cols, low_memory=False)
nw = nw.dropna(subset=['Latitude', 'Longitude', 'Year'])
nw = nw[(nw['Latitude'].between(-90, 90)) & (nw['Longitude'].between(-180, 180)) & ((nw['Latitude'] != 0) | (nw['Longitude'] != 0))]
nw['Year'] = nw['Year'].astype(int)
print(f"{len(nw):,} nest attempts, {nw['Species Name'].nunique()} species, years {nw['Year'].min()}-{nw['Year'].max()}")
counts = nw['Species Name'].value_counts().rename('attempts').to_frame().join(
    nw.drop_duplicates('Species Name').set_index('Species Name')['Species Code'])
counts.loc[SPECIES]

In [ ]:
wanted = {s.lower() for s in SPECIES}
sel = nw[nw['Species Name'].str.lower().isin(wanted) | nw['Species Code'].str.lower().isin(wanted)].copy()
unknown = wanted - set(sel['Species Name'].str.lower()) - set(sel['Species Code'].str.lower())
if unknown:
    raise ValueError(f"Not found in NestWatch: {sorted(unknown)}. Check the spelling against the table above.")
sel = sel[sel['Year'] >= MIN_YEAR]
pool = sel.drop_duplicates('Location ID') if ONE_PER_LOCATION else sel
print(f"{len(sel):,} attempts since {MIN_YEAR} for: {', '.join(sorted(sel['Species Name'].unique()))}; {len(pool):,} in the candidate pool")

## NAIP lookup and chip loading

`get_chip` searches NAIP for the nest's observation year (`search_naip(..., years=[year])`) and reads the chip onto a UTM grid centered on the nest. If a nest has no NAIP survey in its year it returns `None` and another nest is drawn. When several surveys exist for the year (rare, e.g. a scene on a state border) the most recent one is used. The STAC lookup is cached with the chip; "no coverage" results are cached too.

In [ ]:
def get_chip(row):
    """(chip [4,H,W] float32 RGBN, naip_meta dict, Grid) for a nest row, or None if there is no NAIP survey in the nest year."""
    grid = D.Grid.from_center(row['Longitude'], row['Latitude'], SIZE, res=RES)
    key = hashlib.md5(f"{row['Latitude']:.6f}_{row['Longitude']:.6f}_{row['Year']}_{SIZE}_{RES}_v1".encode()).hexdigest()
    path = cache_dir / f"{key}.npz"
    if path.exists():
        z = np.load(path, allow_pickle=True)
        return None if z['chip'].ndim < 3 else (z['chip'], z['meta'].item(), grid)
    try:
        surveys = D.search_naip(grid, years=[row['Year']])
        if not surveys:
            np.savez_compressed(path, chip=np.zeros(0), meta={})
            return None
        items = next(iter(surveys.values()))   # newest survey of that year
        chip = D.read_naip(items, grid)
    except Exception as e:  # network / STAC / COG errors
        warnings.warn(f"{row['Attempt ID']}: {type(e).__name__}: {str(e)[:100]}")
        return None  # not cached, may work on retry
    if not np.isfinite(chip).any():
        np.savez_compressed(path, chip=np.zeros(0), meta={})
        return None
    meta = D.naip_meta(items[0])
    np.savez_compressed(path, chip=chip, meta=meta)
    return chip, meta, grid


def draw_chips(candidates, n):
    """Draw n nests with same-year NAIP at random from `candidates`. Returns [(row, chip, meta, grid)]."""
    candidates = candidates.iloc[rng.permutation(len(candidates))]
    out, tried = [], 0
    with tqdm(total=n, desc='loading NAIP chips') as bar:
        for _, row in candidates.iterrows():
            tried += 1
            res = get_chip(row)
            if res is not None:
                out.append((row, *res))
                bar.update(1)
            if len(out) == n or tried >= MAX_TRIES:
                break
    print(f"checked {tried} nests, {len(out)} with same-year NAIP")
    return out


def to_image(chip, bands):
    """RGBN chip -> display image [H,W,3] in 0-1 with a 2-98% stretch (shared across bands so colors are not distorted)."""
    idx = [0, 1, 2] if bands == 'rgb' else [3, 0, 1]
    img = np.moveaxis(chip[idx], 0, -1)
    lo, hi = np.nanpercentile(img, [2, 98])
    return np.clip((img - lo) / (hi - lo + 1e-6), 0, 1)


def show_chips(drawn):
    n = len(drawn)
    cols = min(COLS, n)
    rows = -(-n // cols)
    fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 4.3 * rows), squeeze=False)
    for ax in axes.flat:
        ax.axis('off')
    for ax, (row, chip, meta, grid) in zip(axes.flat, drawn):
        half = SIZE / 2
        ax.imshow(np.nan_to_num(to_image(chip, BANDS)), extent=(-half, half, -half, half))
        if CIRCLE_RADIUS:
            ax.add_patch(plt.Circle((0, 0), CIRCLE_RADIUS, fill=False, ec='yellow', lw=1, ls='--'))
        ax.plot(0, 0, '+', c='yellow', ms=8)
        ax.set_title(f"{row['Species Name']}\nnest {row['Year']} / NAIP {meta['date']} ({meta['gsd']} m) - {row['Subnational Code']}", fontsize=9)
    fig.tight_layout()
    return fig

## Nest imagery

The center of each chip (yellow +) is the reported nest location. NestWatch coordinates are often the observer-entered location, so expect some positional error: look inside the dashed circle (`CIRCLE_RADIUS`). Titles give species, nest year, NAIP acquisition date and pixel size, and state.

In [ ]:
if PER_SPECIES:
    drawn = [d for _, grp in pool.groupby('Species Name') for d in draw_chips(grp, N_SHOW)]
else:
    drawn = draw_chips(pool, N_SHOW)
if not drawn:
    raise RuntimeError('No nests with same-year NAIP found; try more species, a larger MAX_TRIES, or check network access to Planetary Computer.')
show_chips(drawn)
plt.show()

In [ ]:
# attributes of the nests shown above, with a map link for each
shown = pd.DataFrame([{**r.to_dict(), 'naip_date': m['date'], 'naip_gsd': m['gsd'], 'naip_id': m['id']} for r, _, m, _ in drawn])
shown = shown[['Attempt ID', 'Species Name', 'Year', 'naip_date', 'naip_gsd', 'Subnational Code', 'Latitude', 'Longitude', 'naip_id']].reset_index(drop=True)
shown['map'] = [f"https://www.google.com/maps/search/?api=1&query={a:.6f},{b:.6f}" for a, b in zip(shown['Latitude'], shown['Longitude'])]
shown

In [ ]:
d = {}
d.get('not_key')